In [1]:
# ============================================================
# PHASE 10 — TRAINING INFRASTRUCTURE
# Step 1: Environment and Project Setup
# ============================================================

from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    densenet121,
    DenseNet121_Weights
)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed"
SPLITS_ROOT = DATA_ROOT / "splits"
RESULTS_ROOT = DATA_ROOT / "results"
CHECKPOINT_ROOT = DATA_ROOT / "checkpoints"

CONFIG_ROOT = PROJECT_ROOT / "configs"
DOCS_ROOT = PROJECT_ROOT / "documentation" / "stage_10_training_infrastructure"

# Create required directories
for directory in [
    RESULTS_ROOT,
    CHECKPOINT_ROOT,
    CONFIG_ROOT,
    DOCS_ROOT
]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

CRIC_CROPS = PROCESSED_ROOT / "cric_crops"
CRIC_MANIFEST = PROCESSED_ROOT / "cric_learning_units.csv"
CRIC_SPLITS = SPLITS_ROOT / "cric_splits.csv"

RIVA_CROPS = PROCESSED_ROOT / "riva_crops"
RIVA_MANIFEST = PROCESSED_ROOT / "riva_learning_units.csv"
RIVA_SPLITS = SPLITS_ROOT / "riva_splits.csv"

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# Project training configuration
# ------------------------------------------------------------

NUM_CLASSES = 2
IMAGE_SIZE = 128
BATCH_SIZE = 32

print("=" * 60)
print("PHASE 10 — TRAINING INFRASTRUCTURE")
print("=" * 60)

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024 ** 3),
            2
        ),
        "GB"
    )

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Random seed:", RANDOM_SEED)
print("Selected backbone: DenseNet-121")

print("\nDataset paths:")
print("CRIC crops:", CRIC_CROPS)
print("CRIC manifest:", CRIC_MANIFEST)
print("CRIC splits:", CRIC_SPLITS)

print("\nTraining configuration:")
print("Classes:", NUM_CLASSES)
print("Image size:", f"{IMAGE_SIZE}x{IMAGE_SIZE}")
print("Batch size:", BATCH_SIZE)

print("\nDirectory checks:")
print("CRIC crops exists:", CRIC_CROPS.exists())
print("CRIC manifest exists:", CRIC_MANIFEST.exists())
print("CRIC splits exists:", CRIC_SPLITS.exists())

print("\n=== PHASE 10 SETUP COMPLETE ===")

PHASE 10 — TRAINING INFRASTRUCTURE
Project root: c:\Users\nanda\Documents\cervical-semi-sl
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory: 6.0 GB
PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
Random seed: 42
Selected backbone: DenseNet-121

Dataset paths:
CRIC crops: c:\Users\nanda\Documents\cervical-semi-sl\data\processed\cric_crops
CRIC manifest: c:\Users\nanda\Documents\cervical-semi-sl\data\processed\cric_learning_units.csv
CRIC splits: c:\Users\nanda\Documents\cervical-semi-sl\data\splits\cric_splits.csv

Training configuration:
Classes: 2
Image size: 128x128
Batch size: 32

Directory checks:
CRIC crops exists: True
CRIC manifest exists: True
CRIC splits exists: True

=== PHASE 10 SETUP COMPLETE ===


In [2]:
# ============================================================
# PHASE 10 — Model Factory Test
# ============================================================

import sys

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cervical_ssl.config import (
    BACKBONE,
    NUM_CLASSES,
)

from src.cervical_ssl.models import create_model


model = create_model(
    backbone=BACKBONE,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)

print("=" * 60)
print("MODEL FACTORY TEST")
print("=" * 60)

print("Backbone:", BACKBONE)
print("Number of classes:", NUM_CLASSES)
print("Model:", type(model).__name__)
print("Device:", next(model.parameters()).device)

print("\nClassifier:")
print(model.classifier)

print("\n=== MODEL FACTORY TEST COMPLETE ===")

MODEL FACTORY TEST
Backbone: densenet121
Number of classes: 2
Model: DenseNet
Device: cuda:0

Classifier:
Linear(in_features=1024, out_features=2, bias=True)

=== MODEL FACTORY TEST COMPLETE ===


In [3]:
# ============================================================
# PHASE 10 — Transform Factory Test
# ============================================================

from src.cervical_ssl.transforms import (
    get_train_transform,
    get_val_transform,
)

train_transform = get_train_transform(BACKBONE)
val_transform = get_val_transform(BACKBONE)

print("=" * 60)
print("TRANSFORM FACTORY TEST")
print("=" * 60)

print("Backbone:", BACKBONE)

print("\nTraining transform:")
print(train_transform)

print("\nValidation transform:")
print(val_transform)

print("\n=== TRANSFORM FACTORY TEST COMPLETE ===")

TRANSFORM FACTORY TEST
Backbone: densenet121

Training transform:
Compose(
    Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Validation transform:
Compose(
    Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

=== TRANSFORM FACTORY TEST COMPLETE ===


In [4]:
# ============================================================
# PHASE 10 — Transform Real Image Test
# ============================================================

sample_image_path = CRIC_CROPS / "9ae8a4edde40219bad6303cebc672ee4_cell_1.png"

image = Image.open(sample_image_path).convert("RGB")

train_tensor = train_transform(image)
val_tensor = val_transform(image)

print("=" * 60)
print("REAL IMAGE TRANSFORM TEST")
print("=" * 60)

print("Original image size:", image.size)

print("\nTraining tensor:")
print("Shape:", train_tensor.shape)
print("Dtype:", train_tensor.dtype)

print("\nValidation tensor:")
print("Shape:", val_tensor.shape)
print("Dtype:", val_tensor.dtype)

print("\n=== REAL IMAGE TRANSFORM TEST COMPLETE ===")

REAL IMAGE TRANSFORM TEST
Original image size: (128, 128)

Training tensor:
Shape: torch.Size([3, 128, 128])
Dtype: torch.float32

Validation tensor:
Shape: torch.Size([3, 128, 128])
Dtype: torch.float32

=== REAL IMAGE TRANSFORM TEST COMPLETE ===


In [8]:
# Reload the updated Dataset module

import importlib
import src.cervical_ssl.datasets as datasets_module

importlib.reload(datasets_module)

CervicalDataset = datasets_module.CervicalDataset

print("CervicalDataset reloaded successfully.")

CervicalDataset reloaded successfully.


In [6]:
cric_dataset = CervicalDataset(
    manifest=CRIC_MANIFEST,
    transform=val_transform,
    project_root=PROJECT_ROOT,
)

sample_image, sample_label = cric_dataset[0]

print("=" * 60)
print("DATASET FACTORY TEST")
print("=" * 60)

print("Dataset length:", len(cric_dataset))
print("Image shape:", sample_image.shape)
print("Image dtype:", sample_image.dtype)
print("Label:", sample_label.item())
print("Label dtype:", sample_label.dtype)

print("\n=== DATASET FACTORY TEST COMPLETE ===")

DATASET FACTORY TEST
Dataset length: 11534
Image shape: torch.Size([3, 128, 128])
Image dtype: torch.float32
Label: 1
Label dtype: torch.int64

=== DATASET FACTORY TEST COMPLETE ===


In [9]:
# ============================================================
# PHASE 10 — DataLoader Factory Test
# ============================================================

import importlib
import src.cervical_ssl.datasets as datasets_module

importlib.reload(datasets_module)

CervicalDataset = datasets_module.CervicalDataset
create_dataloader = datasets_module.create_dataloader

cric_dataset = CervicalDataset(
    manifest=CRIC_MANIFEST,
    transform=val_transform,
    project_root=PROJECT_ROOT,
)

cric_loader = create_dataloader(
    dataset=cric_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

images, labels = next(iter(cric_loader))

print("=" * 60)
print("DATALOADER FACTORY TEST")
print("=" * 60)

print("Dataset size:", len(cric_dataset))
print("Batch size:", images.shape[0])
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("Labels in batch:", labels.tolist())

print("\n=== DATALOADER FACTORY TEST COMPLETE ===")

DATALOADER FACTORY TEST
Dataset size: 11534
Batch size: 32
Image batch shape: torch.Size([32, 3, 128, 128])
Label batch shape: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64
Labels in batch: [1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

=== DATALOADER FACTORY TEST COMPLETE ===


In [10]:
# ============================================================
# PHASE 10 — Training Loop Test
# ============================================================

import torch.nn as nn
from torch.optim import AdamW

from src.cervical_ssl.training import (
    train_one_epoch,
    validate_one_epoch,
)

model = create_model(
    backbone=BACKBONE,
    num_classes=NUM_CLASSES,
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.parameters(),
    lr=1e-4,
)

# Use a small subset only for infrastructure testing.
test_dataset = torch.utils.data.Subset(
    cric_dataset,
    range(64),
)

test_loader = create_dataloader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

print("=" * 60)
print("TRAINING LOOP TEST")
print("=" * 60)

train_result = train_one_epoch(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
)

print("\nTraining result:")
print(train_result)

val_result = validate_one_epoch(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=DEVICE,
)

print("\nValidation result:")
print(val_result)

print("\n=== TRAINING LOOP TEST COMPLETE ===")

TRAINING LOOP TEST

Training result:
{'loss': 0.8620719015598297, 'accuracy': 0.375, 'time': 1.7457599639892578}

Validation result:
{'loss': 0.4589255750179291, 'accuracy': 0.828125, 'time': 0.13972997665405273}

=== TRAINING LOOP TEST COMPLETE ===


In [11]:
# ============================================================
# PHASE 10 — Evaluation Test
# ============================================================

from src.cervical_ssl.evaluation import evaluate_model

import src.cervical_ssl.evaluation as evaluation_module
import importlib

importlib.reload(evaluation_module)

evaluate_model = evaluation_module.evaluate_model

evaluation_result = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=DEVICE,
)

print("=" * 60)
print("EVALUATION TEST")
print("=" * 60)

for metric, value in evaluation_result.items():
    print(f"{metric:15s}: {value:.6f}")

print("\n=== EVALUATION TEST COMPLETE ===")

EVALUATION TEST
accuracy       : 0.828125
precision      : 0.938776
recall         : 0.851852
specificity    : 0.700000
macro_f1       : 0.726602
auroc          : 0.925926

=== EVALUATION TEST COMPLETE ===


In [13]:
import importlib
import src.cervical_ssl.checkpoints as checkpoint_module

importlib.reload(checkpoint_module)

save_checkpoint = checkpoint_module.save_checkpoint
load_checkpoint = checkpoint_module.load_checkpoint

print("Checkpoint module reloaded.")

Checkpoint module reloaded.


In [14]:
# ============================================================
# PHASE 10 — Checkpoint Test
# ============================================================

from src.cervical_ssl.checkpoints import (
    save_checkpoint,
    load_checkpoint,
)

import src.cervical_ssl.checkpoints as checkpoint_module
import importlib

importlib.reload(checkpoint_module)

save_checkpoint = checkpoint_module.save_checkpoint
load_checkpoint = checkpoint_module.load_checkpoint


checkpoint_path = CHECKPOINT_ROOT / "phase10_test_checkpoint.pt"

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=1,
    metrics=evaluation_result,
    path=checkpoint_path,
)

print("=" * 60)
print("CHECKPOINT SAVE TEST")
print("=" * 60)

print("Checkpoint path:", checkpoint_path)
print("Checkpoint exists:", checkpoint_path.exists())

# Create a fresh model and optimizer
test_model = create_model(
    backbone=BACKBONE,
    num_classes=NUM_CLASSES,
).to(DEVICE)

test_optimizer = AdamW(
    test_model.parameters(),
    lr=1e-4,
)

checkpoint = load_checkpoint(
    model=test_model,
    optimizer=test_optimizer,
    path=checkpoint_path,
    device=DEVICE,
)

print("\nLoaded checkpoint:")
print("Epoch:", checkpoint["epoch"])
print("Metrics:", checkpoint["metrics"])

print("\n=== CHECKPOINT TEST COMPLETE ===")

CHECKPOINT SAVE TEST
Checkpoint path: c:\Users\nanda\Documents\cervical-semi-sl\data\checkpoints\phase10_test_checkpoint.pt
Checkpoint exists: True

Loaded checkpoint:
Epoch: 1
Metrics: {'accuracy': 0.828125, 'precision': 0.9387755102040817, 'recall': 0.8518518518518519, 'specificity': np.float64(0.7), 'macro_f1': 0.7266019417475729, 'auroc': 0.9259259259259259}

=== CHECKPOINT TEST COMPLETE ===


In [15]:
# ============================================================
# PHASE 10 — Multi-Epoch Training History Test
# ============================================================

import importlib
import src.cervical_ssl.training as training_module

importlib.reload(training_module)

train_model = training_module.train_model

history_model = create_model(
    backbone=BACKBONE,
    num_classes=NUM_CLASSES,
).to(DEVICE)

history_optimizer = AdamW(
    history_model.parameters(),
    lr=1e-4,
)

history = train_model(
    model=history_model,
    train_loader=test_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=history_optimizer,
    device=DEVICE,
    epochs=2,
)

print("\n" + "=" * 60)
print("TRAINING HISTORY TEST")
print("=" * 60)

print("Number of epochs recorded:", len(history))

for record in history:
    print(record)

print("\n=== TRAINING HISTORY TEST COMPLETE ===")

Epoch 1/2 | Train Loss: 0.5730 | Train Acc: 0.7031 | Val Loss: 0.3760 | Val Acc: 0.8750
Epoch 2/2 | Train Loss: 0.2749 | Train Acc: 1.0000 | Val Loss: 0.2603 | Val Acc: 0.9688

TRAINING HISTORY TEST
Number of epochs recorded: 2
{'epoch': 1, 'train_loss': 0.5729555785655975, 'train_accuracy': 0.703125, 'val_loss': 0.3760436475276947, 'val_accuracy': 0.875, 'train_time': 0.778724193572998, 'val_time': 0.14752769470214844}
{'epoch': 2, 'train_loss': 0.27492286264896393, 'train_accuracy': 1.0, 'val_loss': 0.26033513247966766, 'val_accuracy': 0.96875, 'train_time': 0.3235280513763428, 'val_time': 0.14100146293640137}

=== TRAINING HISTORY TEST COMPLETE ===


In [16]:
# ============================================================
# PHASE 10 — Best Checkpoint / Early Stopping Test
# ============================================================

import importlib
import src.cervical_ssl.training as training_module

importlib.reload(training_module)

train_model = training_module.train_model

best_model = create_model(
    backbone=BACKBONE,
    num_classes=NUM_CLASSES,
).to(DEVICE)

best_optimizer = AdamW(
    best_model.parameters(),
    lr=1e-4,
)

best_checkpoint_path = (
    CHECKPOINT_ROOT / "phase10_best_model_test.pt"
)

best_result = train_model(
    model=best_model,
    train_loader=test_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=best_optimizer,
    device=DEVICE,
    epochs=3,
    checkpoint_path=best_checkpoint_path,
    early_stopping_patience=2,
)

print("\n" + "=" * 60)
print("BEST CHECKPOINT TEST")
print("=" * 60)

print("Best epoch:", best_result["best_epoch"])
print("Best validation loss:", best_result["best_val_loss"])
print("Checkpoint exists:", best_checkpoint_path.exists())
print("Epochs recorded:", len(best_result["history"]))

print("\n=== BEST CHECKPOINT TEST COMPLETE ===")

Epoch 1/3 | Train Loss: 0.8676 | Train Acc: 0.3281 | Val Loss: 1.2214 | Val Acc: 0.1562
  → Best checkpoint saved (epoch 1)
Epoch 2/3 | Train Loss: 0.4087 | Train Acc: 0.9531 | Val Loss: 0.7469 | Val Acc: 0.4688
  → Best checkpoint saved (epoch 2)
Epoch 3/3 | Train Loss: 0.2214 | Train Acc: 1.0000 | Val Loss: 0.5138 | Val Acc: 0.7500
  → Best checkpoint saved (epoch 3)

BEST CHECKPOINT TEST
Best epoch: 3
Best validation loss: 0.5137540102005005
Checkpoint exists: True
Epochs recorded: 3

=== BEST CHECKPOINT TEST COMPLETE ===


In [17]:
# ============================================================
# PHASE 10 — Configuration Test
# ============================================================

import importlib
import src.cervical_ssl.config as config_module

importlib.reload(config_module)

print("=" * 60)
print("CONFIGURATION TEST")
print("=" * 60)

print("Backbone:", config_module.BACKBONE)
print("Classes:", config_module.NUM_CLASSES)
print("Image size:", config_module.IMAGE_SIZE)
print("Batch size:", config_module.BATCH_SIZE)
print("Learning rate:", config_module.LEARNING_RATE)
print("Weight decay:", config_module.WEIGHT_DECAY)
print("Epochs:", config_module.EPOCHS)
print("Early stopping patience:", config_module.EARLY_STOPPING_PATIENCE)
print("Random seed:", config_module.RANDOM_SEED)
print("Num workers:", config_module.NUM_WORKERS)
print("Pin memory:", config_module.PIN_MEMORY)

print("\n=== CONFIGURATION TEST COMPLETE ===")

CONFIGURATION TEST
Backbone: densenet121
Classes: 2
Image size: 128
Batch size: 32
Learning rate: 0.0001
Weight decay: 0.0
Epochs: 3
Early stopping patience: 2
Random seed: 42
Num workers: 0
Pin memory: True

=== CONFIGURATION TEST COMPLETE ===


In [19]:
# ============================================================
# PHASE 10 — End-to-End Training Infrastructure Test
# ============================================================

import importlib
import torch.nn as nn
from torch.optim import AdamW

import src.cervical_ssl.config as config_module
import src.cervical_ssl.models as models_module
import src.cervical_ssl.transforms as transforms_module
import src.cervical_ssl.datasets as datasets_module
import src.cervical_ssl.training as training_module
import src.cervical_ssl.evaluation as evaluation_module

importlib.reload(config_module)
importlib.reload(models_module)
importlib.reload(transforms_module)
importlib.reload(datasets_module)
importlib.reload(training_module)
importlib.reload(evaluation_module)


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

backbone = config_module.BACKBONE
num_classes = config_module.NUM_CLASSES
batch_size = config_module.BATCH_SIZE

print("=" * 60)
print("PHASE 10 — END-TO-END INFRASTRUCTURE TEST")
print("=" * 60)


# ------------------------------------------------------------
# Load CRIC split
# ------------------------------------------------------------

cric_df = pd.read_csv(config_module.CRIC_MANIFEST)
cric_split_df = pd.read_csv(config_module.CRIC_SPLITS)

cric_test_split = cric_split_df[
    cric_split_df["split"] == "test"
].reset_index(drop=True)

# Use only a small subset for infrastructure testing.
test_subset = cric_test_split.iloc[:64].copy()


# ------------------------------------------------------------
# Transforms
# ------------------------------------------------------------

train_transform = transforms_module.get_train_transform(
    backbone
)

val_transform = transforms_module.get_val_transform(
    backbone
)


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

test_dataset = datasets_module.CervicalDataset(
    manifest=test_subset,
    transform=val_transform,
    project_root=config_module.PROJECT_ROOT,
)


# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

test_loader = datasets_module.create_dataloader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=config_module.NUM_WORKERS,
    pin_memory=config_module.PIN_MEMORY,
)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = models_module.create_model(
    backbone=backbone,
    num_classes=num_classes,
).to(DEVICE)


# ------------------------------------------------------------
# Loss + optimizer
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.parameters(),
    lr=config_module.LEARNING_RATE,
    weight_decay=config_module.WEIGHT_DECAY,
)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

checkpoint_path = (
    CHECKPOINT_ROOT / "phase10_end_to_end_test.pt"
)

result = training_module.train_model(
    model=model,
    train_loader=test_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    epochs=2,
    checkpoint_path=checkpoint_path,
    early_stopping_patience=2,
)


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

evaluation_result = evaluation_module.evaluate_model(
    model=model,
    dataloader=test_loader,
    device=DEVICE,
)


# ------------------------------------------------------------
# Final infrastructure verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("END-TO-END TEST RESULTS")
print("=" * 60)

print("Backbone:", backbone)
print("Dataset samples:", len(test_dataset))
print("Batch size:", batch_size)
print("Best epoch:", result["best_epoch"])
print("Best validation loss:", result["best_val_loss"])
print("Checkpoint exists:", checkpoint_path.exists())

print("\nEvaluation metrics:")
for metric, value in evaluation_result.items():
    print(f"{metric:15s}: {value:.6f}")

print("\n=== PHASE 10 END-TO-END TEST COMPLETE ===")

PHASE 10 — END-TO-END INFRASTRUCTURE TEST
Epoch 1/2 | Train Loss: 0.9156 | Train Acc: 0.3438 | Val Loss: 0.7738 | Val Acc: 0.3750
  → Best checkpoint saved (epoch 1)
Epoch 2/2 | Train Loss: 0.6128 | Train Acc: 0.6562 | Val Loss: 0.6597 | Val Acc: 0.5625
  → Best checkpoint saved (epoch 2)

END-TO-END TEST RESULTS
Backbone: densenet121
Dataset samples: 64
Batch size: 32
Best epoch: 2
Best validation loss: 0.6597479581832886
Checkpoint exists: True

Evaluation metrics:
accuracy       : 0.562500
precision      : 0.034483
recall         : 1.000000
specificity    : 0.555556
macro_f1       : 0.390476
auroc          : 0.984127

=== PHASE 10 END-TO-END TEST COMPLETE ===
